In [91]:
import numpy as np
from pysr import PySRRegressor

In [92]:
from sympy import Symbol, simplify, expand,symbols, integrate

In [ ]:
import pickle
import torch

# Load the .pkl file
with open('XY_NF_trained.pkl', 'rb') as file:
    data = pickle.load(file)

# Inspect the loaded data
print("Type of data:", type(data))



Type of data: <class 'list'>


In [94]:
data[0].shape

torch.Size([25000, 1, 32, 32])

## Fit PySR

In [ ]:
def extract_patches(field):
    """Extract patches with zero padding for boundary elements, selecting only center, up, down, left, and right."""
    # Pad the field with zeros
    patches = []
    grid_size = field.shape[0]
    padded_field = np.pad(field, pad_width=1, mode='wrap')

    for i in range(grid_size):
        for j in range(grid_size):
            # Extract the center and its immediate neighbors
            
             patch = np.array([
            
                padded_field[i + 1, j + 1],   # Center
                padded_field[i + 1, j + 2],       # Up
                padded_field[i + 2, j + 2],   # Digonal
                padded_field[i + 2, j + 1],    # Right
             
                
               ])
             patches.append(patch)
      
    return np.array(patches)

def extract_patches_for_all(configurations):
    """Extract patches for all configurations."""
    all_patches = []
    for config in configurations:
        patches = extract_patches(config)
        all_patches.extend(patches)
    return np.array(all_patches)

# Step 3: Symbolic Regression using PySR
# # Define a custom loss function
# def custom_loss(predictions, targets):
#     """Mean Squared Error between predicted and true total actions."""
#     return np.mean((predictions - targets) ** 2)
num_conf = 200

patches_all = extract_patches_for_all(((data[0][:,0,:,:][::1][:num_conf].cpu().detach().numpy())))

# Flatten 3x3 patches for input to PySR
X = patches_all
y = np.repeat(np.array(data[1][::1][:num_conf]), 32**2)/8  # Map true actions to each

# Fit the PySR model
model = PySRRegressor(
    niterations=50,  # Number of search iterations
    binary_operators=["+", "-", "*"],
    unary_operators=["square","four(x)=x^4"], # optional to give x^4 operator or not
    loss_function="""
    using Statistics
    function eval_loss(tree, dataset::Dataset{T,L}, options,idx)::L where {T,L}
       # Use batching indices (if provided) to select data
       if isnothing(idx)
           X_batch = dataset.X
           y_batch = dataset.y
       else
           X_batch = dataset.X[:, idx]
           y_batch = dataset.y[idx]
       end
       prediction, flag = eval_tree_array(tree, dataset.X, options)
       #x = dataset.X[:,1]
       #pure, flag = eval_tree_array(tree, x, options)
       if !flag
        return L(Inf)
       end

       window_size = 1024  # Number of patches per configuration
       # Ensure divisibility by window_size
       if length(prediction) % window_size != 0 || length(dataset.y) % window_size != 0
        error("Length of prediction or dataset.y is not divisible by window_size")
       end

       reshaped_pred = reshape(prediction, window_size, :)
       #reshaped_pure = reshape(pure, window_size, :)
       reshaped_y = reshape(dataset.y, window_size, :)

       pred_sums = sum(reshaped_pred, dims=1)
       #pure_sums = sum(reshaped_pure, dims=1)
       y_sums = mean(reshaped_y, dims=1)

       diff = pred_sums .- y_sums
       squared_diff = diff .^ 2
       total_loss = sum(squared_diff) / dataset.n

       return total_loss
    end
    """,
    #progress=True,
    populations = 30,
    maxsize = 30,
    # batching=True,
    # batch_size=128,
    extra_sympy_mappings={
        "four": lambda x : (x**4),
   
    },
    verbosity = 1,
    procs = 16
)


In [96]:
X.shape

(204800, 4)

In [ ]:
# Train the model
model.fit(X,y)

In [74]:
expand(simplify(model.sympy()))

-x0*x3 + 3.0180905*x2**2 - x2*x3 + 0.53505903*x3**4 - 0.16360497

## Compute MSE and R2

In [76]:
def mean_normalize(data):
    return (data - np.mean(data))

In [ ]:
import pickle
import torch

# Load the .pkl file
with open('XY_data.pkl', 'rb') as file:
    data_0 = pickle.load(file)
    if isinstance(data_0[0], torch.Tensor):
      variables = data_0[0].cpu().detach().numpy()  # Move to CPU and convert to NumPy array
    else:
      variables = data_0[0]
    if isinstance(data_0[1], torch.Tensor):
      targets_ = data_0[1].cpu().detach().numpy()  # Move to CPU and convert to NumPy array
    else:
      targets_ = data_0[1]
features = extract_patches_for_all(variables[:,0,:,:])
targets = (mean_normalize(targets_))
wn = 1024

# Inspect the loaded data
print("Type of data:", type(data_0))

Type of data: <class 'list'>


In [78]:
targets = (mean_normalize(targets_))/8


In [79]:
variables.shape

(25000, 1, 32, 32)

In [ ]:
x0, x1, x2, x3, x4, x5 = symbols('x0 x1 x2 x3 x4 x5')

In [83]:
from sympy import Add, expand

In [ ]:
#intrgrate using sympy
expr = (expand(model.equations_.sympy_format[9]))
# Reconstruct the expression
eq = expr#sum(new_terms)
expand(eq)

-x0*x3 + 3.0180905*x2**2 - x2*x3 + 0.53505903*x3**4 - 0.16360497

In [86]:
import sympy as sp
from sklearn.metrics import r2_score, mean_squared_error


In [ ]:
#calculate mse and r2
symbols_  = eq.free_symbols
# Given expression with fewer variables
x0, x1, x2, x3, x4, x5, x6, x7, x8, x9 = sp.symbols('x0 x1 x2 x3 x4 x5 x6 x7 x8 x9')  # Variables in your potential data
# Step 2: Create a mapping from symbol names to data indices
all_symbols = [x0, x1, x2, x3, x4, x5, x6, x7, x8, x9]  # Known order of variables
symbol_index_map = {symbol: i for i, symbol in enumerate(all_symbols)}
# Step 3: Subset data columns matching variables in the expression
relevant_indices = [symbol_index_map[var] for var in symbols_]
subset_data = features[:, relevant_indices]  # Select relevant columns

#globals().update({str(symbol): sp.symbols(str(symbol)) for symbol in symbols})
numerical_function = sp.lambdify(tuple(symbols_), eq, modules="numpy")
# Step 5: Evaluate the expression for each row
y = numerical_function(*subset_data.T)
y_prd =  mean_normalize(y)
# r2_score

mse = np.round(mean_squared_error(y_pred=mean_normalize(np.sum(np.array(y).reshape(-1,wn),axis=1)),y_true=targets),4)
r2 = np.round(r2_score(y_pred=np.sum(np.array(y_prd).reshape(-1,wn),axis=1),y_true=targets),4)
# with open('extra/results.txt', "a") as file:
#     file.write('multivariate_data_for_multi_mean_unit_cov_with_neg_score_through_nf_model_new.txt' + '\n')
#     file.write(str([(eq),f'mse: {mse}',f'r2: {r2}']) + '\n')
print("mse:",mse,"r2:",r2)

mse: 0.0964 r2: 0.9968


In [88]:
subset_data.shape

(25600000, 3)

In [89]:
Equations = []
Equations.append([f"seed = {4}",f"n_samples: {num_conf}", f"population: {20}", f"max_size: {20}", f"iterations: {100}",expand(eq),
                     f"mse: {mse}",
                     f"loss: {r2}"
])
with open("RG/test_data_phi4_32x32_NF_trained.txt", "a") as file:
    for item in Equations:
        file.write(str(item) + "\n")